### **Extraer el `texto`, `autor` y `tags` y exportarlos a un archivo `JSON`**

Vamos a trabajar en el enlace:

https://quotes.toscrape.com/

#### **Crear la spider**

Dado que en el archivo **`1-quotes_to_scrape.ipynb`** realizamos todo el proceso de creación desde el Entorno virtual, instalación de Scrapy y creación del proyecto, ahora solo nos queda comenzar a crear nuestras spiders. Para ello desde la terminal ejecutamos lo siguiente:

<center><img src="https://i.postimg.cc/28hNP6Ww/ws-389.png"></center>

Luego, debemos editar el archivo `quotes_to_scrape_cinco.py` que se crea, con el código que se muestra abajo.

#### **Paso a paso**

Vamos a inspeccionar el sitio:

<center><img src="https://i.postimg.cc/NGP2bPqQ/ws-358.png"></center>

Localizar cada `articulo` de cita:

<center><img src="https://i.postimg.cc/YqGGPfMY/ws-359.png"></center>

Localizar el `texto` de la cita:

<center><img src="https://i.postimg.cc/vZ51YY1m/ws-360.png"></center>

Localizar el `autor` de la cita:

<center><img src="https://i.postimg.cc/vmM4DJ9t/ws-361.png"></center>

Localizar los `tags` de la cita:

<center><img src="https://i.postimg.cc/tCKZGZ4f/ws-362.png"></center>

Localizar si existe un botón de `Siguiente` para pasar a la siguiente página a continuar el proceso:

<center><img src="https://i.postimg.cc/zfjVhxFZ/ws-363.png"></center>

#### **Código**

Veamos ahora nuestra spider modificada para seguir recursivamente el enlace a la página siguiente, extrayendo datos de ella:

In [ ]:
import scrapy


class QuotesToScrapeCincoSpider(scrapy.Spider):
    name = "quotes_to_scrape_cinco"
    allowed_domains = ["quotes.toscrape.com"]
    start_urls = ["https://quotes.toscrape.com"]

    def parse(self, response):
        # Cada vez que queramos localizar un elemento, vamos a utilizar la variable 'response'
        for quote in response.xpath('//div[@class="quote"]'):
            yield {
                'text': quote.xpath('./span[@class="text"]/text()').get(),
                'author': quote.xpath('.//small[@class="author"]/text()').get(),
                'tags': quote.xpath('.//div[@class="tags"]/a[@class="tag"]/text()').getall()
            }

        # Obtenemos un 'Enlace relativo', es decir, obtenemos '/page/2/'
        next_page = response.xpath('//li[@class="next"]/a/@href').get()
        if next_page is not None:
            # Utilizamos '.urljoin()' para unir la URL + next_page  
            # https://quotes.toscrape.com + https://quotes.toscrape.com --> https://quotes.toscrape.com/page/2/
            next_page = response.urljoin(next_page)
            yield scrapy.Request(next_page, callback=self.parse)

Ahora, después de extraer los datos, el método `parse()` busca el enlace a la siguiente página, construye una URL absoluta completa usando el método `urljoin()` (ya que los enlaces pueden ser relativos) y lanza una nueva petición a la siguiente página, registrándose como callback para manejar la extracción de datos para la siguiente página y para mantener el rastreo (crawling) a través de todas las páginas.

Lo que ves aquí es el mecanismo de Scrapy para seguir enlaces: cuando envías una Request (when you yield a Request) en un método callback, Scrapy programará esa request para ser enviada y registrará un método callback para ser ejecutado cuando esa request termine.

Usando esto, puedes construir crawlers complejos que sigan enlaces según las reglas que definas, y extraigan diferentes tipos de datos dependiendo de la página que esté visitando.

En nuestro ejemplo, crea una especie de bucle, siguiendo todos los enlaces a la página siguiente hasta que no encuentra ninguno - útil para rastrear blogs, foros y otros sitios con paginación.

#### **Ejecución**

Nos posicionamos en la ruta correcta y ejecutamos nuestra Spider:

In [ ]:
scrapy crawl quotes_to_scrape_cinco

<center><img src="https://i.postimg.cc/BQFbv4rY/ws-390.png"></center>

De esta manera obtenemos toda la info que pedimos extraer del website:

<center><img src="https://i.postimg.cc/BnG6LRsq/ws-391.png"></center>

Podriamos exportar toda esta data en un archivo `JSON` escribiendo el siguiente comando:

In [ ]:
scrapy crawl quotes_to_scrape_cinco -o quotes_to_scrape_cinco.json

<center><img src="https://i.postimg.cc/BbYycy77/ws-392.png"></center>

#### **Un atajo para crear Requests (Modificación código)**

Como atajo para crear objetos Request puedes utilizar **`response.follow`**:

In [ ]:
import scrapy


class QuotesToScrapeCincoSpider(scrapy.Spider):
    name = "quotes_to_scrape_cinco"
    allowed_domains = ["quotes.toscrape.com"]
    start_urls = ["https://quotes.toscrape.com"]

    # Trabajando con CSS selectors 
    # def parse(self, response):
    #     for quote in response.css("div.quote"):
    #         yield {
    #             "text": quote.css("span.text::text").get(),
    #             "author": quote.css("span small::text").get(),
    #             "tags": quote.css("div.tags a.tag::text").getall(),
    #         }

    #     next_page = response.css("li.next a::attr(href)").get()
    #     if next_page is not None:
    #         yield response.follow(next_page, callback=self.parse)

    def parse(self, response):
        # Cada vez que queramos localizar un elemento, vamos a utilizar la variable 'response'
        for quote in response.xpath('//div[@class="quote"]'):
            yield {
                'text': quote.xpath('./span[@class="text"]/text()').get(),
                'author': quote.xpath('.//small[@class="author"]/text()').get(),
                'tags': quote.xpath('.//div[@class="tags"]/a[@class="tag"]/text()').getall()
            }

        # Obtenemos un 'Enlace relativo', es decir, obtenemos '/page/2/'
        # next_page = response.xpath('//li[@class="next"]/a[@href]').get_attribute('href')  <---- No sirve
        next_page = response.xpath('//li[@class="next"]/a/@href').get()
        if next_page is not None:
            yield response.follow(next_page, callback=self.parse)

A diferencia de `scrapy.Request`, `response.follow` soporta URLs relativas directamente - sin necesidad de llamar a `urljoin`. Tenga en cuenta que `response.follow` sólo devuelve una instancia de `Request`; todavía tiene que producir esta Request.

También puedes pasar un selector a `response.follow` en lugar de una string; este selector debería extraer los atributos necesarios:

In [ ]:
for href in response.xpath('//li[@class="next"]/a/@href'):
    yield response.follow(href, callback=self.parse)

Para los elementos `<a>` existe un atajo: `response.follow` utiliza su atributo `href` automáticamente. Así se puede acortar aún más el código:

In [ ]:
for a in response.xpath('//li[@class="next"]/a'):
    yield response.follow(a, callback=self.parse)

Para crear `múltiples requests` a partir de un iterable, puede utilizar `response.follow_all` en su lugar:

In [ ]:
anchors = response.xpath('//li[@class="next"]/a')
yield from response.follow_all(anchors, callback=self.parse)

o, acortándolo aún más:

In [ ]:
yield from response.follow_all(xpath='//li[@class="next"]/a', callback=self.parse)

#### **Crear una nueva spider y ejecutarla**

Desde la terminal ejecutamos lo siguiente:

<center><img src="https://i.postimg.cc/Pq8Q6q2k/ws-393.png"></center>

Luego, debemos editar el archivo `quotes_to_scrape_seis.py` que se crea, con el nuevo código. Luego, desde la terminal nos posicionamos en la ruta correcta y ejecutamos nuestra Spider:

In [ ]:
scrapy crawl quotes_to_scrape_seis

<center><img src="https://i.postimg.cc/QtGkvryK/ws-394.png"></center>

De esta manera obtenemos toda la info que pedimos extraer del website:

<center><img src="https://i.postimg.cc/cHRMS5p9/ws-395.png"></center>